# Assignment 5: Random Forests

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

cols = ['buying', 'maint', 'doors', "persons", "lug_boot", "safety", "accept"]

df = pd.read_csv("car.data", header=None, names=cols)

attributes = list(df.columns[:-1])
target = df.columns[-1]
X = df[attributes]
y = df[target]

print("Attributes:")
print(X.head())
print()
print("Target:")
print(y.head())

Attributes:
  buying  maint doors persons lug_boot safety
0  vhigh  vhigh     2       2    small    low
1  vhigh  vhigh     2       2    small    med
2  vhigh  vhigh     2       2    small   high
3  vhigh  vhigh     2       2      med    low
4  vhigh  vhigh     2       2      med    med

Target:
0    unacc
1    unacc
2    unacc
3    unacc
4    unacc
Name: accept, dtype: str


In [42]:
class ID3DecisionTree:
    def __init__(self, impurity_metric='entropy', max_depth=None, max_impurity=0.0):
        self.impurity_metric = impurity_metric
        self.max_depth = max_depth
        self.max_impurity = max_impurity
        self.tree = None
        self.accuracy_history = {'train': [], 'val': [], 'test': []}
        self.root_feature = None

    def _get_impurity(self, y):
        if len(y) == 0: return 0
        probs = y.value_counts(normalize=True)
        
        if self.impurity_metric == 'entropy':
            return -np.sum(probs * np.log2(probs + 1e-9))
        elif self.impurity_metric == 'gini':
            return 1 - np.sum(probs**2)
        return 0

    def _information_gain(self, X, y, attribute):
        total_impurity = self._get_impurity(y)
        values = X[attribute].unique()
        weighted_impurity = 0
        
        split_info = 0
        
        for val in values:
            subset_y = y[X[attribute] == val]
            weight = len(subset_y) / len(y)
            weighted_impurity += weight * self._get_impurity(subset_y)
            split_info -= weight * np.log2(weight + 1e-9)

        gain = total_impurity - weighted_impurity
        
        if self.impurity_metric == 'gain_ratio':
            return gain / (split_info + 1e-9)
        return gain

    def fit(self, X, y, depth=0):
        if len(y.unique()) == 1 or (self.max_depth and depth >= self.max_depth) or X.empty:
            return y.mode()[0]

        majority_class = y.mode()[0]

        if not X.columns.any():
            return majority_class
        
        gains = {attr: self._information_gain(X, y, attr) for attr in X.columns}
        if not gains or max(gains.values()) <= 0:
            return majority_class
            
        best_attr = max(gains, key=gains.get)

        subtree_dict = {}
        
        for val in X[best_attr].unique():
            X_subset = X[X[best_attr] == val].drop(columns=[best_attr])
            y_subset = y[X[best_attr] == val]
            if not y_subset.empty:
                subtree_dict[val] = self.fit(X_subset, y_subset, depth + 1)
        
        current_node_tree = (best_attr, majority_class, subtree_dict)
        
        if depth == 0:
            self.tree = current_node_tree
            self.root_feature = best_attr
            
        return current_node_tree

    def predict(self, row, tree=None):
        if tree is None:
            tree = self.tree
        
        if not isinstance(tree, tuple):
            return tree
            
        attr, majority_fallback, subtrees = tree
        val = row.get(attr)
        
        if val in subtrees:
            return self.predict(row, subtrees[val])
        return majority_fallback

    def compute_accuracy(self, data):
        if self.tree is None: return 0
        preds = data.apply(lambda r: self.predict(r), axis=1)
        return (preds == data[target]).mean()

    def _get_feature_depths_recursive(self, node, current_depth, feature_depths):
        if not isinstance(node, tuple):  # Leaf node
            return

        attr, _, subtrees = node
        
        if attr not in feature_depths or current_depth < feature_depths[attr]:
            feature_depths[attr] = current_depth
        
        for val in subtrees:
            self._get_feature_depths_recursive(subtrees[val], current_depth + 1, feature_depths)

    def get_feature_depths(self):
        feature_depths = {}
        if self.tree:
            self._get_feature_depths_recursive(self.tree, 1, feature_depths) # Start depth at 1 for root
        return feature_depths

def prepare_data(df, train_seg, val_seg, percent_noise, with_replacement=False):
    if percent_noise > 0:
        n_samples = int((percent_noise / 100) * df.size)
        for _ in range(n_samples):
            row = np.random.randint(0, df.shape[0])
            col = np.random.randint(0, df.shape[1] - 1)
            col_name = df.columns[col]
            df.iat[row, col] = np.random.choice(df[col_name].unique())

    df_shuffled = df.sample(frac=1, replace=with_replacement).reset_index(drop=True)
    
    train_end = int(len(df) * (train_seg / 100))
    val_end = train_end + int(len(df) * (val_seg / 100))
    
    train = df_shuffled.iloc[:train_end]
    val = df_shuffled.iloc[train_end:val_end]
    test = df_shuffled.iloc[val_end:]
    
    return train, val, test

In [43]:
class RandomForestClassifier:
    def __init__(self, numTrees, featureBagSize, dataBagSize, 
                 impurity_metric='entropy', max_depth=None):
        self.numTrees = numTrees
        self.featureBagSize = featureBagSize
        self.dataBagSize = dataBagSize
        self.impurity_metric = impurity_metric
        self.max_depth = max_depth
        
        self.forest = []
        self.feature_subsets = []
        self.oob_indices_per_tree = []
        self.oob_predictions_per_tree = []
        self.root_features = []
        self.all_feature_depths = []

    def fit(self, X, y):
        num_samples = len(X)
        all_features = list(X.columns)
        
        sample_size = int((self.dataBagSize / 100.0) * num_samples)

        for i in range(self.numTrees):
            tree = ID3DecisionTree(
                impurity_metric=self.impurity_metric, 
                max_depth=self.max_depth
            )
            
            selected_features = np.random.choice(
                all_features, size=self.featureBagSize, replace=False
            ).tolist()
            
            indices = np.random.choice(X.index, size=sample_size, replace=True) 
            
            oob_indices = [idx for idx in X.index if idx not in indices]
            
            X_subset = X.loc[indices, selected_features]
            y_subset = y.loc[indices]
            
            tree.fit(X_subset, y_subset)
            
            self.forest.append(tree)
            self.feature_subsets.append(selected_features)
            self.oob_indices_per_tree.append(oob_indices)
            self.root_features.append(tree.root_feature)
            self.all_feature_depths.append(tree.get_feature_depths())

            if oob_indices:
                oob_X_subset = X.loc[oob_indices, selected_features]
                
                oob_y_preds_for_tree = []
                for _, oob_row in oob_X_subset.iterrows():
                    oob_y_preds_for_tree.append(tree.predict(oob_row))
                
                self.oob_predictions_per_tree.append(
                    pd.DataFrame({'y_true': y.loc[oob_indices], 'y_pred': oob_y_preds_for_tree, 'tree_idx': i})
                )


    def predict_single(self, row):
        predictions = []
        for i in range(self.numTrees):
            tree = self.forest[i]
            features = self.feature_subsets[i]
            
            row_subset = row[features]
            pred = tree.predict(row_subset)
            predictions.append(pred)
        
        votes = Counter(predictions)
        return votes.most_common(1)[0][0]

    def predict(self, X):
        return X.apply(lambda row: self.predict_single(row), axis=1)

    def compute_accuracy(self, X, y):
        preds = self.predict(X)
        return (preds == y).mean()

    def compute_oob_accuracy(self, X_original, y_original):
        if not self.oob_predictions_per_tree:
            return 0.0

        all_oob_preds_df = pd.concat(self.oob_predictions_per_tree)
        
        oob_sample_preds = {}
        for idx in y_original.index:
            sample_oob_preds = all_oob_preds_df[all_oob_preds_df['y_true'].index == idx]
            if not sample_oob_preds.empty:
                votes = Counter(sample_oob_preds['y_pred'])
                oob_sample_preds[idx] = votes.most_common(1)[0][0]

        if not oob_sample_preds:
            return 0.0

        oob_y_true = pd.Series({idx: y_original.loc[idx] for idx in oob_sample_preds.keys()})
        oob_y_pred = pd.Series(oob_sample_preds)

        return (oob_y_true == oob_y_pred).mean()

    def get_root_feature_counts(self):
        root_feature_counts = Counter(self.root_features)
        return dict(root_feature_counts)

    def get_weighted_feature_significance(self, all_available_features):
        weighted_significance = {feature: 0.0 for feature in all_available_features}
        
        for tree_idx, feature_depths in enumerate(self.all_feature_depths):
            if tree_idx >= len(self.feature_subsets):
                continue
            
            current_feature_bag_size = len(self.feature_subsets[tree_idx])

            if current_feature_bag_size == 0:
                continue

            for feature in all_available_features:
                if feature in feature_depths:
                    depth_of_feature = feature_depths[feature]
                    weighted_significance[feature] += (current_feature_bag_size - depth_of_feature + 1) / current_feature_bag_size
        
        return weighted_significance

In [44]:
# Compute stuff in this cell and print in the next cell to avoid unwanted computations
train_df, val_df, test_df = prepare_data(df, train_seg=80, val_seg=10, percent_noise=0)

num_attributes = len(train_df.columns) - 1
rf_model = RandomForestClassifier(
    numTrees=50,
    featureBagSize=4,
    dataBagSize=80,
)

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val =  val_df.drop(columns=[target])
y_val = val_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

rf_model.fit(X_train, y_train)

oob_accuracy = rf_model.compute_oob_accuracy(X_train, y_train)
tr_accr = rf_model.compute_accuracy(X_train, y_train)
v_accr = rf_model.compute_accuracy(X_val, y_val)
ts_accr = rf_model.compute_accuracy(X_test, y_test)

In [45]:
print("--- CLassifier Performance ---")
print(f"Average accuracy for Out-of-Bag data examples: {oob_accuracy * 100:.2f}%")
print(f"Training accuracy: {tr_accr * 100: .2f}%")
print(f"Validation accuracy: {v_accr * 100:.2f}%")
print(f"Testing accuracy: {ts_accr * 100:.2f}%")

print("\n --- Attribute Stats ---")
root_feature_counts = rf_model.get_root_feature_counts()
print("Frequency of features as tree root")
for feature, count in root_feature_counts.items():
    print(f"    - {feature}: {count} times")
    
all_available_features = X_train.columns.tolist()
weighted_significance = rf_model.get_weighted_feature_significance(all_available_features)

print("\n Significance of attributes")
sorted_significance = sorted(weighted_significance.items(), key=lambda item: item[1], reverse=True)
for feature, significance in sorted_significance:
    print(f" - {feature}: {significance: .4f}")

--- CLassifier Performance ---
Average accuracy for Out-of-Bag data examples: 85.02%
Training accuracy:  92.19%
Validation accuracy: 87.21%
Testing accuracy: 89.66%

 --- Attribute Stats ---
Frequency of features as tree root
    - safety: 38 times
    - persons: 11 times
    - buying: 1 times

 Significance of attributes
 - safety:  38.7500
 - persons:  28.2500
 - buying:  21.2500
 - maint:  17.2500
 - lug_boot:  17.0000
 - doors:  10.5000


## Comparison with the Decision Tree Classifier

In [46]:
dt_model = ID3DecisionTree()
dt_model.fit(X_train, y_train)

train_acc = dt_model.compute_accuracy(train_df) * 100
print("Training accuracy:", train_acc)
val_acc = dt_model.compute_accuracy(val_df) * 100
print("Validation accuracy:", val_acc)
test_acc = dt_model.compute_accuracy(test_df) * 100
print("Testing accuracy:", test_acc)

Training accuracy: 100.0
Validation accuracy: 94.18604651162791
Testing accuracy: 94.82758620689656
